# P7: External Hazard Data Ingestion Pipeline — Architecture Design
**Project:** Provable Resilience Intelligence — Lakelink Fiber
**Target Table:** `cdm_tmforum.provable_resilience_intelligence.geographic_hazard_event`
**Current State:** 50,028 synthetic records across 5 hazard types (dig_ticket, wildfire, flood, severe_weather, earthquake)
**Production Goal:** Replace/augment synthetic data with live external feeds

---

## Source APIs & Feeds

| # | Source | Hazard Types | API/Feed | Update Frequency | Auth | Format |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | **NOAA Weather** | severe_weather, flood | `api.weather.gov/alerts/active` | 5-min polling | None (public) | GeoJSON |
| 2 | **USGS Earthquake** | earthquake | `earthquake.usgs.gov/fdsnws/event/1/query` | 1-min polling | None (public) | GeoJSON |
| 3 | **NIFC/InciWeb** | wildfire | `services3.arcgis.com/T4QMspbfLg3qTGWY` (NIFC) | Hourly | None (public) | GeoJSON/REST |
| 4 | **USGS Flood** | flood | `waterwatch.usgs.gov/webservices/floodwatch` | 15-min | None (public) | JSON |
| 5 | **State 811** | dig_ticket | State-specific (varies by Lakelink Fiber service area) | Daily batch | API key per state | CSV/XML |

## Architecture: 3-Stage Ingestion

### Stage 1: API Polling → UC Volume (Bronze)
```
/Volumes/cdm_tmforum/provable_resilience_intelligence/hazard_raw/
  ├── noaa_weather/      dt=YYYY-MM-DD/
  ├── usgs_earthquake/   dt=YYYY-MM-DD/
  ├── nifc_wildfire/     dt=YYYY-MM-DD/
  ├── usgs_flood/        dt=YYYY-MM-DD/
  └── state_811/         state=XX/dt=YYYY-MM-DD/
```
* **Job:** Scheduled Python notebook (Lakeflow Job, serverless)
* **Polling:** `requests.get()` → write JSON to volume as `{source}_{timestamp}.json`
* **Dedup key:** source-specific event ID (NOAA: `properties.id`, USGS: `id`, NIFC: `OBJECTID`, 811: `ticket_number`)
* **Error handling:** Retry 3x with exponential backoff, dead-letter to `/hazard_raw/_failed/`

### Stage 2: Volume → Bronze Delta (Auto Loader SDP)
```sql
CREATE OR REFRESH STREAMING TABLE bronze_hazard_events
AS SELECT * FROM read_files(
  '/Volumes/cdm_tmforum/provable_resilience_intelligence/hazard_raw/',
  format => 'json',
  pathGlobFilter => '*.json',
  recursiveFileLookup => true
)
```
* **Auto Loader:** `cloudFiles` with `schemaEvolutionMode = 'rescue'` for schema drift
* **Partition:** `source`, `ingest_date`

### Stage 3: Bronze → Silver → geographic_hazard_event (SDP)
```sql
CREATE OR REFRESH MATERIALIZED VIEW silver_hazard_events AS
SELECT
  -- Normalize across all sources into PRI schema
  coalesce(noaa_id, usgs_id, nifc_id, ticket_number) AS hazard_id,
  CASE source
    WHEN 'noaa' THEN
      CASE WHEN event LIKE '%Flood%' THEN 'flood'
           WHEN event LIKE '%Fire%' THEN 'wildfire'
           ELSE 'severe_weather' END
    WHEN 'usgs_eq' THEN 'earthquake'
    WHEN 'usgs_flood' THEN 'flood'
    WHEN 'nifc' THEN 'wildfire'
    WHEN '811' THEN 'dig_ticket'
  END AS hazard_type,
  -- Severity mapping per source
  CASE source
    WHEN 'usgs_eq' THEN
      CASE WHEN mag >= 6.0 THEN 'critical'
           WHEN mag >= 4.5 THEN 'high'
           WHEN mag >= 3.0 THEN 'medium'
           ELSE 'low' END
    WHEN 'noaa' THEN
      CASE severity WHEN 'Extreme' THEN 'critical'
           WHEN 'Severe' THEN 'high'
           WHEN 'Moderate' THEN 'medium' ELSE 'low' END
    ...
  END AS severity,
  ST_Y(ST_GeomFromGeoJSON(geometry)) AS latitude,
  ST_X(ST_GeomFromGeoJSON(geometry)) AS longitude,
  estimated_radius_km AS radius_km,
  ST_AsText(ST_GeomFromGeoJSON(geometry)) AS geom_wkt,
  event_start_timestamp AS event_start,
  event_end_timestamp AS event_end,
  source,
  description,
  current_timestamp() AS created_at
FROM LIVE.bronze_hazard_events
```

## Geospatial Processing
* **ST_GeomFromGeoJSON** for source geometries (all APIs return GeoJSON)
* **ST_Distance** / Haversine for span intersection (already implemented in `silver_hazard_span_intersection`)
* **Buffer zones:** NOAA provides `affectedZones` polygons; USGS provides point+magnitude→radius
* **811 dig tickets:** Point geometry + 100m default buffer (excavation zone)

## Scheduling & SLA

| Feed | Polling Interval | Max Staleness | Job Type |
| --- | --- | --- | --- |
| NOAA Weather Alerts | 5 minutes | 10 minutes | Continuous (streaming) |
| USGS Earthquake | 5 minutes | 10 minutes | Continuous (streaming) |
| NIFC Wildfire | 1 hour | 2 hours | Triggered |
| USGS Flood Watch | 15 minutes | 30 minutes | Triggered |
| State 811 | Daily 06:00 UTC | 24 hours | Scheduled |

## Pre-requisites for Production
1. **UC Volume:** Create `/Volumes/cdm_tmforum/provable_resilience_intelligence/hazard_raw/`
2. **Secrets:** 811 API keys stored in Databricks Secrets scope `lakelink-fiber-hazard`
3. **Network:** Outbound HTTPS to `api.weather.gov`, `earthquake.usgs.gov`, `services3.arcgis.com`
4. **SDP Pipeline:** Add bronze/silver stages to existing `resilience_scoring_pipeline` or create dedicated `hazard_ingestion_pipeline`
5. **Monitoring:** Alert on >30min staleness for critical feeds (NOAA, USGS earthquake)

## Current Synthetic Data Disposition
* 50,028 synthetic records serve as test/demo data
* Production: MERGE with dedup on `hazard_id` + `source` — synthetic records get `source='synthetic'` tag
* Migration: Flag synthetic as `event_end = '2026-01-01'` (historical), live data takes over current window

## Data Quality Gates
* Latitude: 24.5–49.0 (continental US)
* Longitude: -125.0 to -66.9
* `hazard_type` IN ('dig_ticket', 'wildfire', 'flood', 'severe_weather', 'earthquake')
* `source` IN ('811_system', 'noaa', 'usgs', 'state_fire_agency', 'nws', 'nifc')
* Dedup: No duplicate `hazard_id` per source within 24h window
* Completeness: latitude, longitude, hazard_type, severity NOT NULL